# Week 62, Day 3: Garbage collection

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w062d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import gc
gc.disable()                         # so the collector can't run by surprise

log = []

class Probe:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        log.append(self.name)

a = Probe("a")
b = a
del a
print(log)
del b
print(log)

x, y = Probe("x"), Probe("y")
x.other, y.other = y, x              # a cycle
del x, y
print(log)
gc.collect()
print(sorted(log))
gc.enable()

In [ ]:
import sys

data = []
before = sys.getrefcount(data)
alias = data
holder = [data, data]
print(sys.getrefcount(data) - before)    # one name and two list slots
del alias, holder
print(sys.getrefcount(data) - before)

log = []
class Probe:
    def __init__(self, name, inner=None):
        self.name, self.inner = name, inner
    def __del__(self):
        log.append(self.name)

chain = Probe("outer", Probe("middle", Probe("inner")))
chain = None                              # outer's count hits zero, which frees the rest
print(log)

In [ ]:
import gc
gc.disable()

log = []
class Node:
    def __init__(self, name):
        self.name, self.parent, self.children = name, None, []
    def add(self, child):
        child.parent = self
        self.children.append(child)
        return child
    def __del__(self):
        log.append(self.name)

root = Node("root")
root.add(Node("leaf"))
del root
print("after del:", log)                 # the parent <-> child cycle keeps both alive
found = gc.collect()
print("after collect:", sorted(log), found > 0)
gc.enable()

In [ ]:
def garbage(objects, outside, edges):
    "objects: names; outside: refs from outside; edges: (from, to) pairs"
    counts = {o: outside.get(o, 0) for o in objects}
    for src, dst in edges:                # every internal reference adds to the real count...
        counts[dst] += 1
    for src, dst in edges:                # ...and the collector subtracts it again
        counts[dst] -= 1
    reachable = [o for o in objects if counts[o] > 0]
    seen = set(reachable)
    while reachable:
        o = reachable.pop()
        for src, dst in edges:
            if src == o and dst not in seen:
                seen.add(dst)
                reachable.append(dst)
    return sorted(set(objects) - seen)

objs = ["main", "cfg", "a", "b"]
links = [("main", "cfg"), ("a", "b"), ("b", "a")]
print(garbage(objs, {"main": 1}, links))

In [ ]:
import gc

class Widget:
    pass

def live(cls):
    return sum(1 for obj in gc.get_objects() if isinstance(obj, cls))

w = Widget()
keeper = {"current": w}
print(live(Widget))
print(any(ref is keeper for ref in gc.get_referrers(w)))   # who holds it? the dict
print(gc.get_referents(keeper) == [w] or w in gc.get_referents(keeper))
print(gc.is_tracked([]), gc.is_tracked(42), gc.is_tracked("text"))
del keeper, w
print(live(Widget))

> **Key idea.** CPython frees an object the moment its reference count reaches zero, which can free a whole chain at once; sys.getrefcount is only good for watching differences. Cycles keep their own counts above zero, so the cyclic collector finds them: it subtracts references inside the tracked group, treats objects with outside references (and what they reach) as alive, and frees the rest, using generations to stay cheap. The gc module runs and switches off the collector and shows referrers, referents and live objects. Break back-pointers by hand to free big structures at once. Python leaks are references you forgot: unbounded caches, registries, observers, closures and stored tracebacks.

### Exercise: Freed at once

A `Link(name, next_link, log)` object appends its name to the list `log` in `__del__`. Write `build_chain(names, log)` that returns the **first** link of a linked list of `Link`s in the given order (each refers to the next), and `drop_from(head, n)`: walk `n` links along the chain and cut it there (set that link's `next_link` to `None`), so reference counting frees everything after it straight away. Return the number of links that remain reachable from `head`.

In [ ]:
class Link:
    def __init__(self, name, next_link, log):
        self.name, self.next_link, self.log = name, next_link, log
    def __del__(self):
        self.log.append(self.name)

def build_chain(names, log):
    ___

def drop_from(head, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_chain():\n    \"reference counting frees chains\"\n    log = []\n    head = build_chain([\"a\", \"b\", \"c\", \"d\"], log)\n    assert head.name == \"a\" and head.next_link.name == \"b\" and log == []\n    assert drop_from(head, 1) == 2 and log == [\"c\", \"d\"]\n    del head\n    assert log == [\"c\", \"d\", \"a\", \"b\"]\n    log2 = []\n    h = build_chain([\"x\"], log2)\n    assert drop_from(h, 0) == 1 and log2 == []\n    assert build_chain([], []) is None")

### Exercise: Simulated reference counting

Simulate reference counting. Write `simulate(ops)` where `ops` is a list of strings: `"new A"` creates object A (count 0), `"bind x A"` makes name `x` refer to A (if `x` already referred to something, that loses a reference first), `"unbind x"` removes the name, `"link A B"` makes A refer to B, and `"unlink A B"` removes one such reference. Whenever an object's count drops to **zero** after it has been referred to, it's freed: record its name, then drop every reference it held (which may free more, depth-first in the order the links were made). Return the list of freed objects in order.

In [ ]:
def simulate(ops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_refcounts():\n    \"reference counting\"\n    ops = [\"new A\", \"new B\", \"new C\", \"bind x A\", \"link A B\", \"link B C\", \"unbind x\"]\n    assert simulate(ops) == [\"A\", \"B\", \"C\"]\n    shared = [\"new A\", \"new B\", \"bind x A\", \"bind y B\", \"link A B\", \"unbind y\", \"unbind x\"]\n    assert simulate(shared) == [\"A\", \"B\"]\n    keep = [\"new A\", \"new B\", \"bind x A\", \"bind y B\", \"link A B\", \"unbind x\"]\n    assert simulate(keep) == [\"A\"]\n    cycle = [\"new A\", \"new B\", \"bind x A\", \"link A B\", \"link B A\", \"unbind x\"]\n    assert simulate(cycle) == []\n    rebind = [\"new A\", \"new B\", \"bind x A\", \"bind x B\", \"bind y B\", \"unbind x\", \"unbind y\"]\n    assert simulate(rebind) == [\"A\", \"B\"]\n    twice = [\"new A\", \"new B\", \"bind x A\", \"link A B\", \"link A B\", \"bind y B\", \"unlink A B\", \"unbind y\"]\n    assert simulate(twice) == []")

### Exercise: Mark and sweep

The simplest way to find garbage is **mark and sweep**: mark everything reachable from the roots, then everything unmarked is garbage. Write `mark(roots, edges)`: the set of objects reachable from the roots, where `edges` is a dict from each object to the list of objects it refers to (iteratively, no recursion), and `sweep(objects, roots, edges)`: the sorted list of objects that are garbage.

In [ ]:
def mark(roots, edges):
    ___

def sweep(objects, roots, edges):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_sweep():\n    \"mark and sweep\"\n    edges = {\"main\": [\"cfg\", \"log\"], \"cfg\": [\"log\"], \"a\": [\"b\"], \"b\": [\"a\"], \"c\": [\"main\"]}\n    assert mark([\"main\"], edges) == {\"main\", \"cfg\", \"log\"}\n    assert sweep([\"main\", \"cfg\", \"log\", \"a\", \"b\", \"c\"], [\"main\"], edges) == [\"a\", \"b\", \"c\"]\n    assert sweep([\"x\"], [], {}) == [\"x\"] and mark([], {}) == set()\n    long = {i: [i + 1] for i in range(10000)}\n    assert len(mark([0], long)) == 10001")

### Exercise: Breaking cycles by hand

`TreeNode` has a parent pointer, so every parent and child form a cycle. Add a method `close()` that breaks **every** cycle in the subtree below (and including) a node, by setting each node's `parent` to `None` and clearing its `children`, without recursion. After `close()`, deleting the last name to the root must free the whole tree **by reference counting alone** (the tests switch the collector off). `size()` returns the number of nodes in the subtree.

In [ ]:
class TreeNode:
    def __init__(self, name, log):
        self.name, self.log = name, log
        self.parent, self.children = None, []

    def add(self, name):
        child = TreeNode(name, self.log)
        child.parent = self
        self.children.append(child)
        return child

    def __del__(self):
        self.log.append(self.name)

    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_detach():\n    \"breaking cycles\"\n    import gc\n    gc.disable()\n    try:\n        log = []\n        root = TreeNode(\"root\", log)\n        a = root.add(\"a\")\n        a.add(\"a1\")\n        root.add(\"b\")\n        del a\n        assert root.size() == 4\n        root.close()\n        del root\n        assert sorted(log) == [\"a\", \"a1\", \"b\", \"root\"]\n        log2 = []\n        r = TreeNode(\"r\", log2)\n        r.add(\"x\")\n        del r\n        assert log2 == []\n        gc.collect()\n        assert sorted(log2) == [\"r\", \"x\"]\n    finally:\n        gc.enable()")

### Exercise: Who is keeping it alive?

Write `live_count(cls)`: how many instances of `cls` the collector is tracking (use `gc.get_objects()`). Then `leaked(func, cls)`: run `func()` once, run `gc.collect()`, and return how many **more** instances of `cls` are alive than before the call. Finally `holders(obj, namespace)`: the sorted list of names in the dict `namespace` whose value is `obj` itself or a list or dict that directly contains `obj` (as an item or a dict value).

In [ ]:
import gc

def live_count(cls):
    ___

def leaked(func, cls):
    ___

def holders(obj, namespace):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_leaks():\n    \"finding leaks\"\n    class Job:\n        pass\n    history = []\n    def leaky():\n        history.append(Job())\n    def tidy():\n        job = Job()\n        return None\n    def cyclic():\n        a, b = Job(), Job()\n        a.other, b.other = b, a\n    assert leaked(leaky, Job) == 1 and leaked(leaky, Job) == 1\n    assert leaked(tidy, Job) == 0 and leaked(cyclic, Job) == 0\n    assert live_count(Job) == 2\n    j = Job()\n    ns = {\"direct\": j, \"in_list\": [1, j], \"in_dict\": {\"k\": j}, \"other\": [Job()], \"copy\": [[j]]}\n    assert holders(j, ns) == [\"direct\", \"in_dict\", \"in_list\"]")

### Bug hunt: A history that never forgets

The log keeps every reading it has ever seen, so its memory grows forever; this prints `1000`. Keep only the most recent three readings, using `collections.deque` with `maxlen`, so it prints `3`.

In [ ]:
from collections import deque

recent = []
for reading in range(1000):
    recent.append(reading)
print(len(recent))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a bounded history\"\n    expect_output(\"3\")")

### Bug hunt: An observer that can't leave

The bus keeps a bound method of the widget, and a bound method refers to its object, so deleting the widget's name doesn't free it: this prints `freed: []`. Unsubscribe the widget's handler before deleting it so it prints `freed: ['w1']`.

In [ ]:
freed = []

class Widget:
    def __init__(self, name):
        self.name = name
    def on_event(self, event):
        pass
    def __del__(self):
        freed.append(self.name)

handlers = []
widget = Widget("w1")
handlers.append(widget.on_event)
del widget
print("freed:", freed)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unsubscribe\"\n    expect_output(\"freed: ['w1']\")")

### Bug hunt: Half a cycle broken

`close` only sets the root's own `parent` to `None` (which it already was), so the cycle between the root and its child is still there and, with the collector off, this prints `freed: []`. Make `close` break the links between the node and its children so it prints `freed: ['child', 'root']`.

In [ ]:
import gc
gc.disable()
freed = []

class Node:
    def __init__(self, name, parent=None):
        self.name, self.parent, self.children = name, parent, []
        if parent:
            parent.children.append(self)
    def close(self):
        self.parent = None
    def __del__(self):
        freed.append(self.name)

root = Node("root")
Node("child", root)
root.close()
del root
print("freed:", sorted(freed))
gc.enable()

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"break the cycle\"\n    expect_output(\"freed: ['child', 'root']\")")

### Puzzle: Counting live instances

Arrange the lines of a function that counts the live instances of a class after a full collection. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: CPython frees an object as soon as its reference count reaches zero
q = ___   # True or False: reference counting alone frees a = []; a.append(a); del a
r = ___   # True or False: gc.disable() also stops reference counting
s = ___   # True or False: gc.is_tracked(42)
t = ___   # True or False: an object that is still reachable but never used again is freed by the collector

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q is False and r is False, \"p, q and r\"\n    assert s is False and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a heap with a cycle collector

Build a simulated heap that combines both of today's mechanisms. `Heap()` has:

- `new(name)`: create an object (count 0); `link(a, b)` / `unlink(a, b)`: a reference from object `a` to object `b` (links can repeat; `unlink` removes one);
- `root(name)` / `unroot(name)`: an outside reference (like a variable) to an object, which can also repeat;
- reference counting: whenever an object's count falls to zero, it's freed at once (and its links released, possibly freeing more); keep a `freed` list in order;
- `collect()`: CPython's algorithm. Copy each live object's count, subtract one for every link between live objects, treat objects whose copy is still above zero as reachable along with everything they reach, and free the rest (add them to `freed` in sorted order, removing their links without counting them as new frees). Return the sorted list of objects it freed;
- `alive()`: the sorted list of live objects; `count(name)`: an object's current reference count.

An object that was created but never referred to stays alive until `collect()` (it's unreachable garbage).

In [ ]:
class Heap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counting():\n    \"reference counting\"\n    h = Heap()\n    for n in \"ABC\":\n        h.new(n)\n    h.root(\"A\")\n    h.link(\"A\", \"B\")\n    h.link(\"B\", \"C\")\n    assert h.count(\"A\") == 1 and h.count(\"B\") == 1\n    h.unroot(\"A\")\n    assert h.freed == [\"A\", \"B\", \"C\"] and h.alive() == []\n    h.new(\"D\")\n    h.root(\"D\")\n    h.root(\"D\")\n    h.unroot(\"D\")\n    assert h.alive() == [\"D\"] and h.count(\"D\") == 1\n\ndef test_cycles():\n    \"the cycle collector\"\n    h = Heap()\n    for n in [\"main\", \"cfg\", \"a\", \"b\", \"c\", \"lonely\"]:\n        h.new(n)\n    h.root(\"main\")\n    h.link(\"main\", \"cfg\")\n    h.link(\"a\", \"b\")\n    h.link(\"b\", \"a\")\n    h.link(\"b\", \"c\")\n    h.root(\"a\")\n    h.unroot(\"a\")\n    assert h.alive() == [\"a\", \"b\", \"c\", \"cfg\", \"lonely\", \"main\"] and h.freed == []\n    assert h.collect() == [\"a\", \"b\", \"c\", \"lonely\"]\n    assert h.alive() == [\"cfg\", \"main\"] and h.freed == [\"a\", \"b\", \"c\", \"lonely\"]\n    h.link(\"cfg\", \"main\")\n    assert h.collect() == []\n    h.unroot(\"main\")\n    assert h.alive() == [\"cfg\", \"main\"] and h.collect() == [\"cfg\", \"main\"]\n    assert h.alive() == []\n\ndef test_reachable_from_cycle():\n    \"reachable through a rooted cycle\"\n    h = Heap()\n    for n in [\"p\", \"q\", \"r\"]:\n        h.new(n)\n    h.link(\"p\", \"q\")\n    h.link(\"q\", \"p\")\n    h.link(\"q\", \"r\")\n    h.root(\"r\")\n    h.root(\"p\")\n    assert h.collect() == [] and h.alive() == [\"p\", \"q\", \"r\"]\n    h.unroot(\"p\")\n    assert h.collect() == [\"p\", \"q\"] and h.alive() == [\"r\"] and h.count(\"r\") == 1")

## Warm-up

`a` is freed the moment its last name goes; the cycle `x <-> y` survives `del` and is freed only when the cycle collector runs.

## Reference counting

Every CPython object carries a **reference count**: how many references point at it (from names, container slots, attributes, the interpreter's own stack). Binding a name, appending to a list or passing an argument adds one; unbinding, removing or returning takes one away. When the count reaches **zero**, the object is freed **immediately**, and that releases its own references, which may free more objects in a chain.

`sys.getrefcount(obj)` shows the count, but the number includes temporary references (the argument itself, among others) and changes between Python versions; some objects (`None`, small ints) are now "immortal" with a fixed huge count. Use it only to watch **differences**, as below.

Reference counting is simple and prompt: a file object you stop using is closed straight away, and memory use stays steady. Its weakness is cycles.

## Cycles

A **reference cycle** is a group of objects that refer to each other: a list containing itself, a parent with a `children` list whose children point back with `parent`, a doubly linked list, an object holding a bound method of itself. When the last outside name goes, every object in the cycle still has a count of at least one (from the others), so reference counting alone would never free them.

Cycles are common and mostly harmless, because CPython has a second mechanism: the **cyclic garbage collector**. But they cost something: objects in a cycle live until the collector next runs, rather than being freed at once, and the collector takes time. Where a structure is big and short-lived, it's worth avoiding cycles (or breaking them, as below) so that reference counting can do the job at once.

## How the cycle collector works

The collector only watches **container** objects (things that can hold references: lists, dicts, class instances, ...), since only they can form cycles. Its trick, roughly:

1. For each tracked object, copy its reference count.
2. For each reference **between** tracked objects, subtract one from the target's copy. What's left is the number of references from **outside** the group (names, the stack, untracked places).
3. Anything with an outside count above zero is definitely reachable; so is everything it reaches (a traversal from those objects).
4. Whatever is left unreached is **garbage**: only cycles keep it alive. It's freed (after calling finalisers such as `__del__`).

To keep this cheap, objects are sorted into **generations**: new objects start in the youngest, and survivors of a collection move to an older one. Most objects die young, so the youngest generation is collected often and the oldest rarely. The thresholds are visible with `gc.get_threshold()`. (The details change between versions: Python 3.14 reorganised the generations into an incremental collector, but the idea of finding unreachable cycles is the same.)

## The gc module

The `gc` module controls and inspects the cycle collector:

- `gc.collect()` runs a full collection now and returns how many unreachable objects it found.
- `gc.disable()` / `gc.enable()` / `gc.isenabled()` turn automatic collection off and on (reference counting carries on regardless). Some programs disable it during a burst of allocation; tests do it to make timing predictable.
- `gc.get_referrers(obj)` and `gc.get_referents(obj)`: what refers to an object, and what it refers to. Handy for asking "**who** is keeping this alive?"
- `gc.get_objects()`: every tracked object, so you can count live instances of a class.
- `gc.is_tracked(obj)`: whether the collector watches it (atoms like ints and strings aren't).
- `gc.freeze()`, `gc.callbacks`, `gc.set_debug(gc.DEBUG_SAVEALL)` for specialised uses.

## Leaks in a garbage-collected language

Garbage collection frees what's **unreachable**. A "leak" in Python is almost always something still **reachable** that you no longer need:

- caches and memo dicts that grow forever (bound them: `functools.lru_cache(maxsize=...)`, `collections.deque(maxlen=...)`);
- module-level lists or registries that objects are added to but never removed from;
- observers and callbacks: a subject's list of bound methods keeps every observer alive (Week 35's warning); unsubscribe, or use weak references (Friday);
- closures and default arguments capturing a large object;
- a stored exception: `e.__traceback__` refers to the frames, which refer to every local variable in them.

The method is always the same: find **who refers** to the object (`gc.get_referrers`, or `tracemalloc` on your own computer to see where memory was allocated) and remove that reference.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, put `import tracemalloc; tracemalloc.start()` at the top of a program you've written and `print(tracemalloc.get_traced_memory())` at the end to see its current and peak memory in bytes. Tomorrow: copying objects properly, with shallow and deep copies.